# Experiment 14: SciBERT Embeddings

**Objective**: Push F1 beyond the 63.33% ceiling by replacing sparse TF-IDF (5k bag-of-words) with dense 768-dim contextual embeddings from `allenai/scibert_scivocab_uncased`. The EDA notebook (53) identified SciBERT as the most likely path to 75%+ F1.

**Strategy**:
- Baseline: Current best (TF-IDF 5k + Venue + Author)
- Config A: Venue + Author features only (no text) — ablation
- Config B: SciBERT-PCA256 + Venue + Author
- Config C: SciBERT-PCA256 only (ablation)
- Config D: Full SciBERT-768 (no PCA) + Venue + Author

**Train/Test split**: 2010–2017 train, 2018–2020 test (same expanded split as Exp 10c which gave 63.33%)

In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import lightgbm as lgb
import pickle

RANDOM_STATE = 42
print('Libraries loaded')

## 1. Load Data

In [ ]:
data_dir = Path('../../data')
feature_dir = data_dir / 'features'

df = pd.read_pickle(data_dir / 'processed/cleaned_data.pkl')

X_all = pd.read_pickle(feature_dir / 'X_all.pkl')
y_cls = pd.read_pickle(feature_dir / 'y_classification.pkl')
metadata = pd.read_pickle(feature_dir / 'metadata.pkl')

print(f'Cleaned data: {df.shape}')
print(f'Features (X_all): {X_all.shape}')
print(f'Target: {y_cls.shape}')
print(f'\nClass distribution: {y_cls.value_counts().to_dict()}')
print(f'High-impact rate: {y_cls.mean():.1%}')

## 2. Temporal Split (2010–2017 train, 2018–2020 test)

In [ ]:
train_years = list(range(2010, 2018))
test_years  = [2018, 2019, 2020]

train_mask = df['Year'].isin(train_years)
test_mask  = df['Year'].isin(test_years)

train_idx = df[train_mask].index.intersection(X_all.index)
test_idx  = df[test_mask].index.intersection(X_all.index)

inst_col = next((c for c in df.columns if 'institution' in c.lower() and 'num' not in c.lower()), None)
if inst_col:
    aub_mask = df[inst_col].str.contains('AUB|American University of Beirut', case=False, na=False)
    train_idx = df[train_mask & aub_mask].index.intersection(X_all.index)
    test_idx  = df[test_mask & aub_mask].index.intersection(X_all.index)
    print(f'Using AUB-only filter (col: {inst_col})')
else:
    print('No institution column found — using all papers')

y_train = y_cls.loc[train_idx]
y_test  = y_cls.loc[test_idx]

print(f'\nTrain: {len(train_idx)} papers ({y_train.mean():.1%} high-impact)')
print(f'Test:  {len(test_idx)} papers ({y_test.mean():.1%} high-impact)')

## 3. Build Feature Sets

Separate structured features (venue + author) from TF-IDF text features.

In [ ]:
tfidf_cols = [c for c in X_all.columns if c.startswith('tfidf_')]
struct_cols = [c for c in X_all.columns if not c.startswith('tfidf_')]

print(f'TF-IDF features:     {len(tfidf_cols)}')
print(f'Structured features: {len(struct_cols)}')
print(f'Structured feature names: {struct_cols}')

X_struct_train = X_all.loc[train_idx, struct_cols].copy()
X_struct_test  = X_all.loc[test_idx,  struct_cols].copy()

X_tfidf_train = X_all.loc[train_idx]
X_tfidf_test  = X_all.loc[test_idx]

print(f'\nFull feature matrix (TF-IDF + structured): {X_tfidf_train.shape}')

## 4. SciBERT Abstract Embeddings

Use `allenai/scibert_scivocab_uncased` to encode paper abstracts into dense 768-dim vectors.
Embeddings are cached to disk so this cell only runs once.

In [ ]:
EMBEDDING_CACHE = feature_dir / 'scibert_embeddings.pkl'
SCIBERT_MODEL   = 'allenai/scibert_scivocab_uncased'

def encode_abstracts_scibert(texts, model_name=SCIBERT_MODEL, batch_size=32, max_length=512):
    """Encode abstracts with SciBERT using mean-pooling over token embeddings."""
    import torch
    from transformers import AutoTokenizer, AutoModel

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')
    print(f'Loading {model_name}...')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []
    texts = [str(t) if pd.notna(t) else '' for t in texts]

    from tqdm import tqdm
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=max_length, return_tensors='pt').to(device)
        with torch.no_grad():
            output = model(**encoded)
        attention_mask = encoded['attention_mask'].unsqueeze(-1).float()
        token_embeddings = output.last_hidden_state
        mean_emb = (token_embeddings * attention_mask).sum(1) / attention_mask.sum(1)
        all_embeddings.append(mean_emb.cpu().numpy())

    return np.vstack(all_embeddings)


if EMBEDDING_CACHE.exists():
    print(f'Loading cached embeddings from {EMBEDDING_CACHE}')
    with open(EMBEDDING_CACHE, 'rb') as f:
        embedding_data = pickle.load(f)
    scibert_embeddings = embedding_data['embeddings']
    embedding_index    = embedding_data['index']
    print(f'Loaded embeddings: {scibert_embeddings.shape}')
else:
    print('Computing SciBERT embeddings...')
    all_idx_ordered = list(train_idx) + list(test_idx)
    abstracts = df.loc[all_idx_ordered, 'Abstract'].tolist()
    scibert_embeddings = encode_abstracts_scibert(abstracts)
    embedding_index    = all_idx_ordered
    feature_dir.mkdir(parents=True, exist_ok=True)
    with open(EMBEDDING_CACHE, 'wb') as f:
        pickle.dump({'embeddings': scibert_embeddings, 'index': embedding_index}, f)
    print(f'Saved to {EMBEDDING_CACHE}')
    print(f'Embeddings shape: {scibert_embeddings.shape}')

In [ ]:
idx_to_pos = {idx: pos for pos, idx in enumerate(embedding_index)}

train_positions = [idx_to_pos[i] for i in train_idx]
test_positions  = [idx_to_pos[i] for i in test_idx]

emb_train_raw = scibert_embeddings[train_positions]   # (n_train, 768)
emb_test_raw  = scibert_embeddings[test_positions]    # (n_test, 768)

print(f'Train embeddings: {emb_train_raw.shape}')
print(f'Test  embeddings: {emb_test_raw.shape}')

N_COMPONENTS = 256
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
emb_train_pca = pca.fit_transform(emb_train_raw)
emb_test_pca  = pca.transform(emb_test_raw)

explained = pca.explained_variance_ratio_.cumsum()[-1]
print(f'\nPCA {N_COMPONENTS} components explain {explained:.1%} of variance')

emb_cols = [f'scibert_{i}' for i in range(N_COMPONENTS)]
df_emb_train = pd.DataFrame(emb_train_pca, index=train_idx, columns=emb_cols)
df_emb_test  = pd.DataFrame(emb_test_pca,  index=test_idx,  columns=emb_cols)

## 5. Evaluate Configurations

Helper function to train and evaluate a model with threshold optimization.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, label=''):
    """Train model, optimize threshold, and return F1/AUC."""
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)

    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.30, 0.76, 0.01):
        preds = (proba >= thr).astype(int)
        f1 = f1_score(y_test, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    preds = (proba >= best_thr).astype(int)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)

    print(f'{label:55s}  F1={best_f1:.4f}  AUC={auc:.4f}  P={prec:.4f}  R={rec:.4f}  thr={best_thr:.2f}')
    return {'label': label, 'f1': best_f1, 'auc': auc, 'precision': prec, 'recall': rec, 'threshold': best_thr}

results = []

### Config 0 — Baseline (TF-IDF 5k + Venue + Author)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_tfidf_train.fillna(0), X_tfidf_test.fillna(0), y_train, y_test,
                   label='Baseline: TF-IDF 5k + Venue + Author (LR)')
results.append(r)

### Config A — Venue + Author only (no text) — ablation

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_struct_train.fillna(0), X_struct_test.fillna(0), y_train, y_test,
                   label='Config A: Venue + Author only (LR)')
results.append(r)

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_struct_train.fillna(0), X_struct_test.fillna(0), y_train, y_test,
                   label='Config A: Venue + Author only (LightGBM)')
results.append(r)

### Config B — SciBERT-PCA256 + Venue + Author

In [ ]:
X_full_train = pd.concat([df_emb_train, X_struct_train.fillna(0)], axis=1)
X_full_test  = pd.concat([df_emb_test,  X_struct_test.fillna(0)],  axis=1)

print(f'Combined feature matrix: {X_full_train.shape}')

lr = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1,
                        C=1.0, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author (LR)')
results.append(r)

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author (LightGBM)')
results.append(r)

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
r = evaluate_model(rf, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author (RF)')
results.append(r)

### Config C — SciBERT-PCA256 only (ablation)

In [ ]:
lr = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
r = evaluate_model(lr, df_emb_train, df_emb_test, y_train, y_test,
                   label='Config C: SciBERT-PCA256 only (LR)')
results.append(r)

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, df_emb_train, df_emb_test, y_train, y_test,
                   label='Config C: SciBERT-PCA256 only (LightGBM)')
results.append(r)

### Config D — Full SciBERT-768 (no PCA) + Venue + Author

In [ ]:
emb_cols_full = [f'scibert_full_{i}' for i in range(768)]
df_emb_train_full = pd.DataFrame(emb_train_raw, index=train_idx, columns=emb_cols_full)
df_emb_test_full  = pd.DataFrame(emb_test_raw,  index=test_idx,  columns=emb_cols_full)

X_full768_train = pd.concat([df_emb_train_full, X_struct_train.fillna(0)], axis=1)
X_full768_test  = pd.concat([df_emb_test_full,  X_struct_test.fillna(0)],  axis=1)

print(f'Full 768-dim feature matrix: {X_full768_train.shape}')

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_full768_train, X_full768_test, y_train, y_test,
                   label='Config D: SciBERT-768 + Venue + Author (LightGBM)')
results.append(r)

## 6. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1', ascending=False)

baseline_f1 = results_df[results_df['label'].str.startswith('Baseline')]['f1'].values[0]
best_prev   = 0.6333  # Exp 10c selective domain segmentation

print('=' * 100)
print('FINAL RESULTS SUMMARY — Experiment 14: SciBERT Embeddings')
print('=' * 100)
print(f'{"Config":<55}  {"F1":>8}  {"vs TF-IDF baseline":>20}  {"vs best-ever (63.33%)": >22}')
print('-' * 100)
for _, row in results_df.iterrows():
    delta_base = (row['f1'] - baseline_f1) * 100
    delta_best = (row['f1'] - best_prev) * 100
    marker = ' <- BEST' if row['f1'] == results_df['f1'].max() else ''
    print(f"{row['label']:<55}  {row['f1']*100:>7.2f}%  {delta_base:>+19.2f} pp  {delta_best:>+21.2f} pp{marker}")

print('=' * 100)
best = results_df.iloc[0]
print(f'\nBEST CONFIG: {best["label"]}')
print(f'  F1:        {best["f1"]*100:.2f}%')
print(f'  AUC:       {best["auc"]*100:.2f}%')
print(f'  Precision: {best["precision"]*100:.2f}%')
print(f'  Recall:    {best["recall"]*100:.2f}%')
print(f'  Threshold: {best["threshold"]:.2f}')
print(f'\n  vs TF-IDF baseline: {(best["f1"] - baseline_f1)*100:+.2f} pp')
print(f'  vs best-ever 63.33%: {(best["f1"] - best_prev)*100:+.2f} pp')